In [ ]:
from crewai import Agent, Task, Process, Crew, LLM
from dotenv import load_dotenv
import os
import asyncio
import nest_asyncio

nest_asyncio.apply()

load_dotenv()

model = os.getenv('MODEL_NAME', 'glm-5.2')
api_base = os.getenv('OPENAI_API_BASE', '')

def build_travel_crew(destination: str, days: int, budget: float, interests: str) -> str:
    llm = LLM(model=model, temperature=0.4, api_base=api_base)

    researcher = Agent(
        role="目的地研究员",
        goal=f"研究{destination}并提供关键的旅行建议",
        backstory="资深旅游记者,到访过 100 多个国家,熟知最好的小众景点和实用建议。",
        llm=llm,
        verbose=True,
    )

    planner = Agent(
        role="旅行行程规划师",
        goal=f"为{destination}制定一份详细的{days}天行程",
        backstory="拥有 15 年经验的奢华旅行顾问,擅长打造个性化行程。",
        llm=llm,
        verbose=True,
    )

    budget_analyst = Agent(
        role="旅行预算分析师",
        goal=f"在${budget}预算内估算切合实际的旅行成本",
        backstory="财务旅行顾问,帮助旅行者在预算内最大化体验。",
        llm=llm,
        verbose=True,
    )

    research_task = Task(
        description=f"""研究{destination}作为一次{days}天旅行的目的地。
        涵盖:最佳旅行时间、适合住宿的街区、必看景点、当地美食、交通建议,
        以及需要了解的文化习俗。
        旅行者兴趣:{interests}""",
        agent=researcher,
        expected_output="目的地概要,包含重点区域、景点、美食和实用建议",
    )

    planning_task = Task(
        description=f"""为{destination}制定一份{days}天的行程。
        预算:总计${budget}。兴趣:{interests}。
        包含上午/下午/晚上的活动、具体的餐厅推荐,以及地点之间的交通时间。确保行程可行且令人愉快。""",
        agent=planner,
        expected_output=f"按天划分的{days}天行程,包含活动、餐饮和时间安排",
        context=[research_task],
    )

    budget_task = Task(
        description=f"""为这趟{days}天的{destination}之行提供预算明细。
        总预算:${budget}。包含:机票(估算)、住宿、餐饮、活动、交通。
        若预算紧张请标出,并提出调整建议。""",
        agent=budget_analyst,
        expected_output="逐项预算明细,含每日均值和省钱建议",
        context=[research_task, planning_task],
    )

    crew = Crew(
        agents=[researcher, planner, budget_analyst],
        tasks=[research_task, planning_task, budget_task],
        process=Process.sequential,
        verbose=True,
        tracing=True
    )

   
    result = asyncio.run(crew.kickoff_async())
    return str(result)

In [3]:
itinerary = build_travel_crew(destination='北京', days=7, budget=3000, interests="历史文化底蕴")

print("=" * 60)
print("🗺️  TRAVEL ITINERARY")
print("=" * 60)
print(itinerary)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 745532b1-d3be-47cc-8f7c-693d350cbf69                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 研究北京作为一次7天旅行的目的地。                                                                        │
│          涵盖:最佳旅行时间、适合住宿的街区、必看景点、当地美食、交通建议,                                       │
│          以及需要了解的文化习俗。                                                                               │
│          旅行者兴趣:历史文化底蕴                                                                                │
│  ID: 5e6beddf-3ca6-48c4-88d6-69126e1c00b5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 目的地研究员                                                                                            │
│                                                                                                                 │
│  Task: 研究北京作为一次7天旅行的目的地。                                                                        │
│          涵盖:最佳旅行时间、适合住宿的街区、必看景点、当地美食、交通建议,                                       │
│          以及需要了解的文化习俗。                                                                               │
│          旅行者兴趣:历史文化底蕴                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 目的地研究员                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 北京7日历史文化深度旅行指南                                                                                  │
│                                                                                                                 │
│  ## 📍 目的地概要                                                                                               │
│  北京，元明清三朝古都，联合国教科文组织“北京中轴线”世界文化遗产所在地。它不仅是紫禁城、长城、天坛的符号集合，   │
│  更是由胡同肌理、坛庙礼制、皇家园林、近代风云与市井烟火共同编织的活态文化容器。对于痴迷历史底蕴的旅行者，7天行  │
│  程应摒弃打卡式漫游，以“中轴线申遗脉络”为骨架，串联宫廷、祭祀、陵寝、宗教与市井空间，辅以专家级动线规划与避峰   │
│  策略，实现从“看风景”到“读城市”的跃迁。                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 🌤 最佳旅行时间                                                                                              │
│  | 季节 | 推荐指数 | 气候特征 | 旅行建议 |                                                                      │
│  |------|----------|----------|----------|                                                                      │
│  | **4月中-5月下旬** | ★★★★★ | 15-25℃，湿度适中，玉兰/海棠/国槐次第开花 |                                       │
│  避开五一黄金周首尾两日；春季多浮尘，备N95口罩与保湿霜 |                                                        │
│  | **9月中旬-10月下旬** | ★★★★★ | 秋高气爽，昼夜温差10℃左右，银杏/黄栌变色 |                                    │
│  国庆后第二周起游客锐减，机票酒店价格回落30%+ |                                                                 │
│  | **6-8月** | ★★★☆☆ | 高温多雨（均温28-32℃），偶有雷阵雨 | 适合夜游（前门、亮马河、首钢园）；室内场馆避暑首选  │
│  |                                                                                                              │
│  | **11月-次年3月** | ★★★★☆ | 干冷，均温-5-5℃，雪后故宫/长城出片率极高 |                                        │
│  需厚羽绒服+防滑鞋；部分户外路段结冰，长城建议选慕田峪/司马台 |                                                 │
│                                                                                                                 │
│  **避峰提示：**                                                                                                 │
│  避开全国两会（3月初）、国庆长假（10.1-7）、春节假期（1月下旬-2月中旬）及大型展会期，否则核心景点预约难度呈指   │
│  数级上升。                                                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 🏡 适合住宿的街区推荐                                                                                       │
│  | 街区 | 文化契合度 | 代表业态 | 推荐类型/酒店参考 | 通勤优势 |                                                │
│  |------|------------|----------|-------------------|----------|                                                │
│  | **前门-大栅栏-珠市口（西城/东城交界）** | ★★★★★ | 老字号聚集、胡同改造民宿、天桥曲艺 |                       │
│  四合雅苑、金台别墅、漫心酒店（前门店） | 距正阳门、国家大剧院、天桥步行10分钟内，地铁2/7/8号线交汇 |           │
│  | **鼓楼-什刹海片区** | ★★★★☆ | 钟鼓楼晨昏仪式、水系景观、独立书店/咖啡馆 |                                    │
│  平谷小院、北平机器主题空间、华堂公寓 | 地铁2/8号线鼓楼大街站，骑行可达南锣鼓巷、景山 |                         

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 旅行预算分析师                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 📊 北京7日历史文化深度旅行·预算明细与财务优化方案                                                            │
│                                                                                                                 │
│  ## 💰 一、 总体预算分配表（总计 $3,000）                                                                       │
│  | 预算大类 | 分配金额 | 占比 | 财务测算依据 | 现实可行性评估 |                                                 │
│  |----------|----------|------|----------------|----------------|                                               │
│  | **机票（往返）** | $350 | 11.7% | 国内主流城市经济舱往返均价；国际/港澳台航线需上浮$100-200 | ✅             │
│  合理。提前30-45天购票可锁定此区间 |                                                                            │
│  | **住宿（7晚）** | $1,050 | 35.0% | 前门/鼓楼片区精品民宿或中端设计酒店，均$150/晚（含早） | ⚠️               │
│  旺季（4月/10月）易涨至$180-200/晚，属预算敏感项 |                                                              │
│  | **餐饮（7天）** | $600 | 20.0% | 日均$85，结构：2顿市井小吃($15×2)+1顿特色正餐($55) | ✅                     │
│  合理。避开景区内部溢价餐厅即可稳定控制 |                                                                       │
│  | **交通（全周期）** | $300 | 10.0% | 市内地铁/公交/共享单车$50；机场接驳$30；跨区专线/拼车$220 | ⚠️           │
│  若改用“专属商务车”将超支$200+，需严格管控 |                                                                    │
│  | **活动/门票/讲解/体验** | $550 | 18.3% |                                                                     │
│  核心景点联票$180；官方讲解器/志愿者团$80；1项深度体验$150；杂费$140 | ✅ 合理。依赖公共资源可最大化体验密度 |  │
│  | **预备金/保险/应急** | $150 | 5.0% | 基础旅游险$15；伴手礼/特产直邮$40；医疗/票务差价缓冲$95 | ✅            │
│  必要。应对临时改签、天气变动或物价波动 |                                                                       │
│  | **合计** | **$3,000** | **100%** | 已预留5%安全垫，无隐性消费，符合中高端文化游定位 | ✅ 预算闭环完整 |      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 📅 二、 每日预算均值与执行节奏                                                                              │
│  | 日期 | 行程主题 | 当日预估花费 | 费用构成拆解 | 财务优化节点 |                                               │
│  |------|----------|--------------|----------------|----------------|                                           │
│  | **D1** | 中轴起点·皇城气象 | $380 | 住宿$150+餐饮$80+交通$30+故宫门票/讲解$120 |                             │
│  故宫选非景观位午餐省$30；东华门步道免费替代付费城墙骑行 |                                                      │
│  | **D2** | 礼制巅峰·天人合一 | $350 | 住宿$150+餐饮$80+交通$20+天坛/先农坛门票/讲解$100 |                      │
│  天坛租官方智能讲解器($5)替代私人导游；永定门外部免费观景 |                                                     │
│  | **D3** | 山水园林·皇家别业 | $360 | 住宿$150+餐饮$80+交通$30+颐和园/圆明园门票$100 |                         │
│  颐和园14:00后团队散客锐减，可购半日通票；圆明园西洋楼区免费开放 |                                              │
│  | **D4** | 边关雄风·长城寻迹 | $480 | 住宿$150+餐饮$70+交通$120(慕田峪专线)+门票/缆车/滑道$140 |               │
│  放弃包车改乘“德胜门旅游专线”或正规一日游大巴，单程$3-5，立省$80 |                                              │
│  | **D5** | 宗教艺术·市井烟火 | $340 | 住宿$150+餐饮$80+交通$20+雍和宫/孔庙/胡同体验$90 |                       │
│  雍和宫香花券免费；五道营独立茶室人均$8替代高价融合菜 |                                                         │
│  | **D6** | 陵寝秘境·地下宫殿 | $420 | 住宿$150+餐饮$70+交通$100(昌平公交/市郊铁路)+十三陵联票/讲解$100 |       │
│  十三陵联票比单买长陵+定陵省$20；居庸关云海观景台免费替代收费栈道 |            

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 为这趟7天的北京之行提供预算明细。                                                                        │
│          总预算:$3000。包含:机票(估算)、住宿、餐饮、活动、交通。                                                │
│          若预算紧张请标出,并提出调整建议。                                                                      │
│  Agent: 旅行预算分析师                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🗺️  TRAVEL ITINERARY
# 📊 北京7日历史文化深度旅行·预算明细与财务优化方案

## 💰 一、 总体预算分配表（总计 $3,000）
| 预算大类 | 分配金额 | 占比 | 财务测算依据 | 现实可行性评估 |
|----------|----------|------|----------------|----------------|
| **机票（往返）** | $350 | 11.7% | 国内主流城市经济舱往返均价；国际/港澳台航线需上浮$100-200 | ✅ 合理。提前30-45天购票可锁定此区间 |
| **住宿（7晚）** | $1,050 | 35.0% | 前门/鼓楼片区精品民宿或中端设计酒店，均$150/晚（含早） | ⚠️ 旺季（4月/10月）易涨至$180-200/晚，属预算敏感项 |
| **餐饮（7天）** | $600 | 20.0% | 日均$85，结构：2顿市井小吃($15×2)+1顿特色正餐($55) | ✅ 合理。避开景区内部溢价餐厅即可稳定控制 |
| **交通（全周期）** | $300 | 10.0% | 市内地铁/公交/共享单车$50；机场接驳$30；跨区专线/拼车$220 | ⚠️ 若改用“专属商务车”将超支$200+，需严格管控 |
| **活动/门票/讲解/体验** | $550 | 18.3% | 核心景点联票$180；官方讲解器/志愿者团$80；1项深度体验$150；杂费$140 | ✅ 合理。依赖公共资源可最大化体验密度 |
| **预备金/保险/应急** | $150 | 5.0% | 基础旅游险$15；伴手礼/特产直邮$40；医疗/票务差价缓冲$95 | ✅ 必要。应对临时改签、天气变动或物价波动 |
| **合计** | **$3,000** | **100%** | 已预留5%安全垫，无隐性消费，符合中高端文化游定位 | ✅ 预算闭环完整 |

---

## 📅 二、 每日预算均值与执行节奏
| 日期 | 行程主题 | 当日预估花费 | 费用构成拆解 | 财务优化节点 |
|------|----------|--------------|----------------|----------------|
| **D1** | 中轴起点·皇城气象 | $380 | 住宿

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 745532b1-d3be-47cc-8f7c-693d350cbf69                                                                       │
│  Final Output: # 📊 北京7日历史文化深度旅行·预算明细与财务优化方案                                              │
│                                                                                                                 │
│  ## 💰 一、 总体预算分配表（总计 $3,000）                                                                       │
│  | 预算大类 | 分配金额 | 占比 | 财务测算依据 | 现实可行性评估 |                                                 │
│  |----------|----------|------|----------------|----------------|                                               │
│  | **机票（往返）** | $350 | 11.7% | 国内主流城市经济舱往返均价；国际/港澳台航线需上浮$100-200 | ✅             │
│  合理。提前30-45天购票可锁定此区间 |                                                                            │
│  | **住宿（7晚）** | $1,050 | 35.0% | 前门/鼓楼片区精品民宿或中端设计酒店，均$150/晚（含早） | ⚠️               │
│  旺季（4月/10月）易涨至$180-200/晚，属预算敏感项 |                                                              │
│  | **餐饮（7天）** | $600 | 20.0% | 日均$85，结构：2顿市井小吃($15×2)+1顿特色正餐($55) | ✅                     │
│  合理。避开景区内部溢价餐厅即可稳定控制 |                                                                       │
│  | **交通（全周期）** | $300 | 10.0% | 市内地铁/公交/共享单车$50；机场接驳$30；跨区专线/拼车$220 | ⚠️           │
│  若改用“专属商务车”将超支$200+，需严格管控 |                                                                    │
│  | **活动/门票/讲解/体验** | $550 | 18.3% |                                                                     │
│  核心景点联票$180；官方讲解器/志愿者团$80；1项深度体验$150；杂费$140 | ✅ 合理。依赖公共资源可最大化体验密度 |  │
│  | **预备金/保险/应急** | $150 | 5.0% | 基础旅游险$15；伴手礼/特产直邮$40；医疗/票务差价缓冲$95 | ✅            │
│  必要。应对临时改签、天气变动或物价波动 |                                                                       │
│  | **合计** | **$3,000** | **100%** | 已预留5%安全垫，无隐性消费，符合中高端文化游定位 | ✅ 预算闭环完整 |      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 📅 二、 每日预算均值与执行节奏                                                                              │
│  | 日期 | 行程主题 | 当日预估花费 | 费用构成拆解 | 财务优化节点 |                                               │
│  |------|----------|--------------|----------------|----------------|                                           │
│  | **D1** | 中轴起点·皇城气象 | $380 | 住宿$150+餐饮$80+交通$30+故宫门票/讲解$120 |                             │
│  故宫选非景观位午餐省$30；东华门步道免费替代付费城墙骑行 |                                                      │
│  | **D2** | 礼制巅峰·天人合一 | $350 | 住宿$150+餐饮$80+交通$20+天坛/先农坛门票/讲解$100 |                      │
│  天坛租官方智能讲解器($5)替代私人导游；永定门外部免费观景 |                                                     │
│  | **D3** | 山水园林·皇家别业 | $360 | 住宿$150+餐饮$80+交通$30+颐和园/圆明园门票$100 |                         │
│  颐和园14:00后团队散客锐减，可购半日通票；圆明园西洋楼区免费开放 |                                              │
│  | **D4** | 边关雄风·长城寻迹 | $480 | 住宿$150+餐饮$70+交通$120(慕田峪专线)+门票/缆车/滑道$140 |               │
│  放弃包车改乘“德胜门旅游专线”或正规一日游大巴，单程$3-5，立省$80 |                                              │
│  | **D5** | 宗教艺术·市井烟火 | $340 | 住宿$150+餐饮$80+交通$20+雍和宫/孔庙/胡同体验$90 |                       │
│  雍和宫香花券免费；五道营独立茶室人均$8替代高价融合菜 |                                                         │
│  | **D6** | 陵寝秘境·地下宫殿 | $420 | 住宿$150+餐饮$70+交通$100(昌平公交/市郊铁路)+十三陵联票/讲解$100 |       │
│  十三陵联票比单买长陵+定陵省$20；居庸关云海观景台免费替代收费栈道 |    

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯